# Big-M排除(tight M・Indicator) — 適用前・原理・適用・効果

固定費(施設開設・段取り替えなど)を「開設したら生産できる」という論理制約でモデル化するとき、
`x <= M*y`(Big-M)の M をどう選ぶかで LP 緩和の強さが激変する。M が緩いと、y がわずかに
正の分数値を取るだけで x がほぼフルに使えてしまい、下界がほとんど締まらない。

この notebook は次の型でこの現象と対処を追う。

1. **課題(before)** — 緩い Big-M で `mk.analyze` に掛け、純粋LP緩和境界の弱さを確認する
2. **原理(principle)** — M の大小で LP 緩和領域がどう変わるかを図で見る
3. **適用(how)** — tight M と `addConsIndicator`(SCIP Indicator制約)を当てる
4. **効果(before/after)** — `mk.compare_variants` でルート双対境界・gap・ノード数を比較する

題材は **固定費付き生産計画**(`samples/others/fixed_charge.py`)。8施設、各施設は
開設(`y_i=1`)して初めて生産`x_i>0`できる。`build_model(bigm=...)` が `"loose"`(巨大定数)・
`"tight"`(実際に可能な最大生産量)・`"indicator"`(SCIP Indicator制約)の3変種を提供する。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import fixed_charge as fc

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 課題(before) — 緩い Big-M を診断する

まず緩い Big-M 版(`bigm="loose"`)を `mk.analyze` に掛ける。

In [2]:
report = mk.analyze(lambda: fc.build_model("loose"), name="fixed_charge-loose", time_limit=10)
print(report.summary())

[fixed_charge-loose] 検出症状 1件:
  [good] 制約-変数がブロック構造 + 少数の結合制約 -> ベンダーズ分解 / Dantzig-Wolfe分解(結合制約を主問題に)


この規模(8施設)は presolve が緩い Big-M を自動でタイト化してしまうため、
`numerical_scale` は発火**しない**(残存係数比は presolve 後で見る設計 — [3. Big-M排除](../../playbook/03-bigm.md)
の「効かないとき」参照)。診断だけでは見えない弱さを、presolve/cut を切った**純粋LP緩和境界**で
直接確認する。

In [3]:
for bigm in ("loose", "tight", "indicator"):
    print(f"{bigm:10s}: 純粋LP緩和境界 = {fc.lp_relaxation_bound(bigm):.1f}")

loose     : 純粋LP緩和境界 = 1594.2
tight     : 純粋LP緩和境界 = 7126.7
indicator : 純粋LP緩和境界 = 7126.7


緩い Big-M は LP 緩和境界が大きく低い(下界が弱い)。presolve/cut を有効にした通常求解では
この差の多くが自動で埋められてしまうが、「定式化そのものの質」としては歴然とした差がある。
次でその原理を図で見る。

## 2. 原理(principle) — Big-M の緩さが LP 緩和領域を広げる

施設1つの連動制約 `x <= M*y`(`y` はバイナリの連続緩和 `0<=y<=1`、`x` は `0<=x<=cap`)を考える。
M が真に必要な値(`cap`、tight)のときは、`y` を少し正にしても `x` はその分しか使えない
(領域は原点から `(cap, 1)` への直線の下側)。M が巨大(loose)だと、`y` がごくわずかに
正であるだけで `x` は上限 `cap` まで使えてしまう — 「わずかな開設コストでフル生産」という
現実にありえない解が LP 緩和では許されてしまう。

In [4]:
cap = 70.0  # F1の容量
tight_m = 70.0  # min(容量, 総需要) = tight M
mids = [500.0, 5000.0]
loose_m = 100000.0

def hex_to_rgba(h, alpha):
    h = h.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

y = np.linspace(0, 1, 400)
Ms = [(tight_m, "tight M = 70 (真に必要な最小値)", C["s1"]),
      (500.0, "M = 500", C["s2"]),
      (5000.0, "M = 5000", C["warn"]),
      (loose_m, "loose M = 100000", C["muted"])]

fig = go.Figure(layout=base_layout(
    "連動制約 x <= M・y の LP緩和領域(y:施設開設の連続緩和, x:生産量)",
    "y(開設、連続緩和)", "x(生産量、上限cap=70)", height=440))
for m_val, label, col in Ms:
    upper = np.minimum(cap, m_val * y)
    fig.add_trace(go.Scatter(x=np.r_[y, y[::-1]], y=np.r_[upper, np.zeros_like(y)],
        fill="toself", fillcolor=hex_to_rgba(col, 0.10),
        line=dict(color=col, width=2.5), name=label,
        hovertemplate=label + ": x<=%{y:.1f}<extra></extra>"))
fig.update_yaxes(range=[0, cap * 1.05])
# y=0.01 という「ほぼ閉鎖」でもloose Mならxがフル生産できてしまう点を強調
fig.add_annotation(x=0.02, y=cap * 0.98, ax=0.25, ay=cap * 0.98,
    text="y=0.02 でも loose M なら x はほぼ cap まで使える(緩和の「ただ乗り」)",
    showarrow=True, arrowhead=2, arrowcolor=C["warn"],
    font=dict(color=C["warn"], size=11))
show(fig)

tight M(青)は原点から `(1, cap)` への直線そのもので、`y` の値に生産量が厳密に比例する
「無駄のない」緩和になる。M が大きくなるほど(緑→橙→グレー)、曲線は左上に張り付いて
`y` がほぼ0でも `x` がフルに使える領域が広がる — これが LP 緩和境界を下げる正体。

**Indicator制約はこの M 選びの問題自体をなくす。** `y=0 ⟹ x<=0` という論理をソルバーに直接渡し、
分枝時にこの論理制約を扱う専用の分離処理(indicator cut)で締める。M の値をユーザーが選ぶ
必要がなくなる。

## 3. 適用(how) — tight M と `addConsIndicator`

汎用ヘルパーは無く、PySCIPOpt を直接使う。

```python
# tight M: 変数境界から導出できる最小のMに絞る
m.addCons(x[i] <= min(cap[i], demand) * y[i], name=f"link_{i}")

# Indicator制約: y=0 のとき x<=0(activeone=False で「binvar=0のとき制約が有効」の向きにする)
m.addConsIndicator(x[i] <= 0, binvar=y[i], activeone=False, name=f"link_{i}")
```

下は最小の動作確認。3つの定式化がすべて同じ最適値に到達することを確認する。

In [5]:
from pyscipopt import Model, quicksum

def toy_model(bigm: str) -> Model:
    m = Model(); m.hideOutput()
    cap, need = 10.0, 4.0
    x = m.addVar(lb=0, ub=cap, name="x")
    y = m.addVar(vtype="B", name="y")
    m.addCons(x >= need, name="demand")  # 需要4を満たすには x>0、よってリンク制約で y=1 が必須
    if bigm == "loose":
        m.addCons(x <= 10000.0 * y, name="link")
    elif bigm == "tight":
        m.addCons(x <= cap * y, name="link")
    else:
        m.addConsIndicator(x <= 0, binvar=y, activeone=False, name="link")
    m.setObjective(5 * y + 2 * x, "minimize")
    return m

for bigm in ("loose", "tight", "indicator"):
    mm = toy_model(bigm)
    mm.optimize()
    print(f"{bigm:10s}: obj={mm.getObjVal():.2f}  status={mm.getStatus()}")

loose     : obj=13.00  status=optimal
tight     : obj=13.00  status=optimal
indicator : obj=13.00  status=optimal


3変種とも同じ最適値に到達する(等価な論理制約の異なる表現)。差は緩和の締まり方であり、
最適値そのものではない。

## 4. 効果(before/after) — `mk.compare_variants`

`fixed_charge` の3変種(loose / tight / indicator)を同じ時間制限で解き、
**ルート双対境界・最終gap・ノード数**を比較する。`mk.compare_variants` は既定設定
(presolve/cutあり)で測るので、まずはそれをそのまま見る。

In [6]:
df = mk.compare_variants(
    {"loose Big-M":  lambda: fc.build_model("loose"),
     "tight Big-M":  lambda: fc.build_model("tight"),
     "indicator":    lambda: fc.build_model("indicator")},
    time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,loose Big-M,7180.0,7138.333333,0.008639,1
1,tight Big-M,7180.0,7138.333333,0.008639,1
2,indicator,7180.0,7136.476721,0.008901,1


**正直な結果**: この8施設の規模では、既定設定の `root_dual`/`final_gap`/`nodes` は
3変種でほぼ**同じ**になる。1節で見た通り `numerical_scale` も発火しなかった通り、
**presolveが緩いBig-Mを自動でタイト化してしまう**ため(playbook「効かないとき」参照)。
「定式化そのものの質」の差は、presolve/cutを切った**純粋LP緩和境界**(1節で計算済み)で
初めて見える。

In [7]:
labels = ["loose Big-M", "tight Big-M", "indicator"]
bar_colors = [C["muted"], C["s1"], C["s2"]]
rows = [df.loc[df["variant"] == lab].iloc[0] for lab in labels]
pure_lp = [fc.lp_relaxation_bound(b) for b in ("loose", "tight", "indicator")]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.10,
    subplot_titles=("純粋LP緩和境界(presolve/cut off) — 定式化そのものの質",
                    "既定設定でのルート双対境界(presolve/cut あり)",
                    "既定設定での探索ノード数"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, pure_lp, lambda v: f"{v:.0f}")
add_bars(2, [r["root_dual"] for r in rows], lambda v: f"{v:.0f}")
add_bars(3, [r["nodes"] for r in rows], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="Big-M排除の before / after — 定式化の質 vs 既定設定での実測", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [8]:
loose, tight, ind = rows
print(f"純粋LP緩和境界(presolve/cut off): loose={pure_lp[0]:.1f} -> tight={pure_lp[1]:.1f} "
      f"(+{(pure_lp[1]/pure_lp[0]-1)*100:.0f}%) -> indicator={pure_lp[2]:.1f}")
print(f"既定設定ルート双対境界(presolve/cut あり): loose={loose['root_dual']:.1f} "
      f"-> tight={tight['root_dual']:.1f} -> indicator={ind['root_dual']:.1f}  (ほぼ同じ)")
print(f"既定設定ノード数: loose={int(loose['nodes'])} -> tight={int(tight['nodes'])} "
      f"-> indicator={int(ind['nodes'])}")

純粋LP緩和境界(presolve/cut off): loose=1594.2 -> tight=7126.7 (+347%) -> indicator=7126.7
既定設定ルート双対境界(presolve/cut あり): loose=7180.0 -> tight=7180.0 -> indicator=7180.0  (ほぼ同じ)
既定設定ノード数: loose=1 -> tight=1 -> indicator=1


## まとめ

- **純粋LP緩和境界(presolve/cut off)は tight M / indicator で大きく上がる**(1594→7127、
  +347%)。loose Big-M は「y がわずかに正なら x がフル生産できる」緩和を許し、下界を
  大きく下げていた。これが「定式化そのものの質」の差である。
- 一方、**既定設定(presolve/cutあり)でのルート双対境界・ノード数はほぼ同じ**になった
  (このリポジトリの規模では presolve が差を吸収してしまうため)。「定式化の質」と
  「既定設定での実測」は必ずしも同じ図では見えない、という点自体が実務上の教訓である。
- tight M と indicator の差は多くの場合小さい — indicator の強みは「M の値を選ぶ必要が
  なくなる」ことそのもの(選定ミスのリスクを構造的に排除する)。

### なぜ SCIP が自動でやらないのか

**部分的にはやる。** 既定の presolve は緩い Big-M をある程度自動でタイト化するため、
このリポジトリの規模(8施設)では最終的な求解時間・ノード数の差は数字ほど大きくない
(素朴なB&Bのノード数は11→9→8程度に留まる、[FINDINGS §3](../../playbook/03-bigm.md))。
差が実感できるのは presolve が効きにくい大規模/複雑な構造のときである。SCIP が完全に
自動化できない理由は、「本当に必要な最小M」の算出には**モデルの意味論**(このリンクで
本当に許容されるxの最大値は何か)が要ることが多く、presolveの汎用ルールだけでは
tight Mまで詰め切れないケースが残るため。indicator制約はこの限界を構造的に回避する。

### 効かないとき・注意

- `numerical_scale` は presolve **後**の残存係数比で判断する設計。presolve前の生の係数比
  (loose Big-Mなら1e5オーダー)だけを見て「悪い」と早合点しないこと。
- 施設数が少なく presolve が効きやすいモデルでは、tight M 化の効果は「LP緩和境界の改善」
  としては明確でも、最終求解時間の改善としては小さいことがある。

関連: [プレイブック 3. Big-M排除](../../playbook/03-bigm.md) /
[プレイブック 8. 条件数・数値健全性](../../playbook/08-condition.md)
(`08_condition_number.ipynb` で同じ loose/tight Big-M の条件数 κ(A) を比較する)。